# 11 — Zbritja e gradientit dhe optimizimi numerik

        **Pyetja udhëheqëse:** Si lëvizim në hapësirën e parametrave drejt një minimumi?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të interpretohet gradienti gjeometrikisht.
- Të implementohet gradient descent.
- Të studiohet roli i learning rate.
- Të prezantohet zbritja stokastike e gradientit.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("11 — Zbritja e gradientit dhe optimizimi numerik")
print("Seed:", RANDOM_STATE)

# 1. Një sipërfaqe humbjeje — CORE

Zbritja e gradientit (gradient descent) përdor
$$
\theta_{n+1}=\theta_n-\eta\nabla L(\theta_n).
$$
Gradienti tregon drejtimin e rritjes më të shpejtë; prandaj $-\nabla L$ tregon drejtimin lokal të
zbritjes më të shpejtë.

In [ ]:
def loss(th):
    x,y=th
    return 0.5*(4*x*x + y*y)
def grad(th):
    x,y=th
    return np.array([4*x,y])

xx,yy=np.meshgrid(np.linspace(-3,3,200),np.linspace(-3,3,200))
zz=0.5*(4*xx**2+yy**2)
fig,ax=plt.subplots()
ax.contour(xx,yy,zz,levels=20)
ax.set_xlabel("θ1"); ax.set_ylabel("θ2"); ax.set_title("Sipërfaqja e humbjes")
plt.show()

# 2. Trajektorja e optimizimit — CORE

In [ ]:
def gd(theta0,eta,n=50):
    th=np.array(theta0,dtype=float); hist=[th.copy()]
    for _ in range(n):
        th=th-eta*grad(th); hist.append(th.copy())
    return np.array(hist)

hist=gd([2.5,2.5],0.15,40)
fig,ax=plt.subplots()
ax.contour(xx,yy,zz,levels=20)
ax.plot(hist[:,0],hist[:,1],marker="o",markersize=3)
ax.set_title("Trajektorja e zbritjes së gradientit")
plt.show()

### Optimizer-i i ripërdorshëm nga `src/`

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))
from src.optimization.gradient_descent import gradient_descent
theta_src, hist_src = gradient_descent([2.5,2.5], grad, lr=0.15, n_iter=40)
print("Minimumi nga src:", theta_src)

# 3. Learning rate — CORE

In [ ]:
etas=[0.02,0.15,0.55]
fig,ax=plt.subplots()
for eta in etas:
    h=gd([2.5,2.5],eta,35)
    vals=[loss(t) for t in h]
    ax.semilogy(vals,label=f"η={eta}")
ax.set_xlabel("Iteracioni"); ax.set_ylabel("L")
ax.legend(); ax.set_title("Shumë i vogël, i përshtatshëm, shumë i madh")
plt.show()

# 4. Regresion linear me gradient descent — CORE

In [ ]:
x=np.linspace(-2,2,120)
y=1.8*x+0.6+rng.normal(0,0.35,len(x))
A=np.c_[np.ones(len(x)),x]
theta=np.zeros(2); eta=0.08; hist=[]
for _ in range(250):
    pred=A@theta
    g=2*A.T@(pred-y)/len(y)
    theta-=eta*g
    hist.append(theta.copy())
theta

In [ ]:
fig,ax=plt.subplots()
ax.scatter(x,y,s=12)
ax.plot(x,A@theta)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Regresion i trajnuar me gradient descent")
plt.show()

# 5. Trajektorja e parametrave — CORE

In [ ]:
H=np.array(hist)
fig,ax=plt.subplots()
ax.plot(H[:,0],H[:,1],marker=".",markersize=3)
ax.set_xlabel("β0"); ax.set_ylabel("β1")
ax.set_title("Trajektorja në hapësirën e parametrave")
plt.show()

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Zbritja stokastike e gradientit — EXTENDED

Në zbritjen stokastike të gradientit (stochastic gradient descent, SGD), gradienti vlerësohet nga
një mostër ose mini-batch. Kjo ul koston për hap dhe fut stokasticitet në trajektore.

In [ ]:
theta=np.zeros(2); eta=0.03; path=[]
for epoch in range(12):
    for i in rng.permutation(len(x)):
        ai=A[i]
        err=ai@theta-y[i]
        theta-=eta*2*err*ai
        path.append(theta.copy())
P=np.array(path)
fig,ax=plt.subplots()
ax.plot(P[:,0],P[:,1],alpha=0.7)
ax.set_xlabel("β0"); ax.set_ylabel("β1")
ax.set_title("Trajektore stokastike e SGD")
plt.show()

# 7. Aplikim jo-fizik: optimizim i një modeli ekonomik — EXTENDED

E njëjta logjikë numerike përdoret në optimizimin e modeleve në ekonomi, biologji dhe inxhinieri. Diferenca kryesore është kuptimi i funksionit të humbjes, jo algoritmi i optimizimit.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Gjeni teorikisht kufirin e stabilitetit për η në funksionin kuadratik të dhënë.
2. Krahasoni numrin e iteracioneve për η të ndryshme.
3. Implementoni early stopping kur norma e gradientit bie nën një prag.
4. Krahasoni batch GD me SGD.

        ## B. Ushtrim i plotë
        Trajnoni regresionin logjistik me gradient descent dhe vizualizoni humbjen për tre learning rates.

        ## C. Playground
        Ndryshoni learning rate, pikën fillestare, batch size dhe numrin e iteracioneve.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.